# VoiceForge -- Day 10 (part 1): Generate 3-Way Evaluation Outputs

**Goal for today:** generate base / SFT-only / SFT+DPO completions for
**every** held-out test prompt (not just a handful), plus a rule-based
compliance check per variant, so Day 10 part 2 (a local script) can turn
this into a pairwise win-rate and rubric scores.

Loads base model once and attaches both adapters as named adapters on the
same model, switching between them with `set_adapter()` -- avoids loading
the base model three separate times.

**Before running:** `Runtime > Change runtime type > T4 GPU`, `HF_TOKEN`
Colab secret set.


## 1. Install dependencies

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets huggingface_hub


## 2. GPU check

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU detected -- set Runtime > Change runtime type > T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))
major, minor = torch.cuda.get_device_capability()
USE_BF16 = major >= 8
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"Compute capability: {major}.{minor} -> using {'bf16' if USE_BF16 else 'fp16'}")


## 3. Mount Drive, log in, load the held-out test set

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/voiceforge"
Path(DRIVE_PROJECT_DIR).mkdir(parents=True, exist_ok=True)


In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get("HF_TOKEN"))

from datasets import load_dataset

SFT_DATASET_REPO = "bikalpoudel/voiceforge-brand-voice-sft"
sft_ds = load_dataset(SFT_DATASET_REPO)
test_set = sft_ds["test"]
print(f"{len(test_set)} held-out test prompts (never seen by SFT or DPO training)")


## 4. Load base model + both adapters as named, switchable adapters

`set_adapter("sft")` -> SFT-only behavior. `set_adapter("dpo")` -> SFT+DPO
behavior (the DPO adapter's weights already incorporate the SFT starting
point, since Day 9 continued training the same LoRA weights rather than
stacking a second adapter). `model.disable_adapter()` -> base behavior.
One loaded base model, three comparable variants.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
SFT_ADAPTER_REPO = "bikalpoudel/voiceforge-brand-voice-sft-lora"
DPO_ADAPTER_REPO = "bikalpoudel/voiceforge-brand-voice-dpo-lora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=COMPUTE_DTYPE,
)

model = PeftModel.from_pretrained(base_model, SFT_ADAPTER_REPO, adapter_name="sft")
model.load_adapter(DPO_ADAPTER_REPO, adapter_name="dpo")
model.eval()
model.config.use_cache = True

print(f"Loaded base + 2 named adapters. Memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


## 5. System prompts (same as Day 6/7/9)

In [ ]:
DISTILLED_SYSTEM_PROMPTS = {
    "cozy_crochet": (
        "You are a copywriter for a cozy handmade crochet shop. Voice: genuine, "
        "cozy, cute, simple, thoughtful. Short sentences, contractions and "
        "fragments are fine. Emoji: choose from \U0001F9F6 \U0001F338 \u2728 \U0001F49B, max 1-2 per post. Never say: "
        "elevate, premium, luxurious, game-changer, exclusive drop. Include one "
        "concrete detail (size, turnaround time, or material) when the format "
        "calls for it. Output only the copy itself, no labels or quotation marks."
    ),
    "romantic_floral": (
        "You are a copywriter for a romantic handmade bouquet shop. Voice: "
        "sentimental, heartfelt, aesthetic, warm, custom-focused. Emoji: choose "
        "from \U0001F490 \U0001F380 \U0001F48C \u2728, max 1-2 per post. Never say: unlock, revolutionary, "
        "unrivaled, cheap, bulk, flash sale. Include one concrete detail (size, "
        "turnaround time, or materials) when the format calls for it. Output "
        "only the copy itself, no labels or quotation marks."
    ),
}


## 6. Rule-based compliance checks (same rules used since Day 4)

In [ ]:
import re

CLICHE_BLOCKLIST = {
    "cozy_crochet": [
        "elevate your accessory game", "must-have staple for your wardrobe",
        "unleash your style", "the ultimate fashion statement",
        "luxurious", "premium quality", "synergy", "game-changer",
        "game changer", "exclusive drop",
    ],
    "romantic_floral": [
        "unlock the secrets of romance", "unrivaled elegance for your lifestyle",
        "revolutionary floral technology", "cheap", "standard", "bulk",
        "commercial floral", "flash sale",
    ],
}
FORMAT_WORD_BOUNDS = {
    "ig_caption": (12, 55), "tiktok_hook": (2, 16), "tiktok_caption": (8, 40),
    "etsy_listing": (25, 90), "promo_announcement": (4, 28), "restock_announcement": (8, 40),
}
CTA_KEYWORDS = [
    "dm", "message us", "message me", "send us", "send a", "order now",
    "order today", "shop now", "shop the", "link in bio", "comment below",
    "swipe up", "tap", "visit", "book your", "reach out",
]
CTA_REQUIRED_FORMATS = {"ig_caption", "etsy_listing", "restock_announcement"}
CTA_FORBIDDEN_FORMATS = {"tiktok_hook"}

def contains_any(text_lower, phrases):
    return [p for p in phrases if re.search(r"\b" + re.escape(p) + r"\b", text_lower)]

def check_candidate(text, voice, fmt):
    issues = []
    wc = len(text.split())
    lo, hi = FORMAT_WORD_BOUNDS.get(fmt, (1, 10_000))
    if wc < lo or wc > hi:
        issues.append(f"length_out_of_range(words={wc})")
    hits = contains_any(text.lower(), CLICHE_BLOCKLIST.get(voice, []))
    if hits:
        issues.append(f"banned_phrase({','.join(hits)})")
    has_cta = bool(contains_any(text.lower(), CTA_KEYWORDS))
    if fmt in CTA_REQUIRED_FORMATS and not has_cta:
        issues.append("missing_required_cta")
    if fmt in CTA_FORBIDDEN_FORMATS and has_cta:
        issues.append("cta_present_but_forbidden")
    return issues


## 7. Generate all 3 variants for every test prompt

Resumable: safe to stop and rerun, already-generated ids are skipped.


In [ ]:
import json

OUT_PATH = f"{DRIVE_PROJECT_DIR}/eval_generations.jsonl"

def generate(voice, brief, max_new_tokens=150):
    messages = [{"role": "system", "content": DISTILLED_SYSTEM_PROMPTS[voice]}, {"role": "user", "content": brief}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=True,
            temperature=0.8, top_p=0.9, pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def already_done_ids(path):
    if not Path(path).exists():
        return set()
    return {json.loads(l)["id"] for l in Path(path).read_text().splitlines() if l.strip()}

done = already_done_ids(OUT_PATH)
print(f"{len(done)} of {len(test_set)} already generated. Resuming...")

with open(OUT_PATH, "a", encoding="utf-8") as f:
    for i, row in enumerate(test_set):
        if row["id"] in done:
            continue

        with model.disable_adapter():
            base_out = generate(row["voice"], row["brief"])
        model.set_adapter("sft")
        sft_out = generate(row["voice"], row["brief"])
        model.set_adapter("dpo")
        dpo_out = generate(row["voice"], row["brief"])

        out_row = {
            "id": row["id"], "voice": row["voice"], "voice_tag": row["voice_tag"],
            "format": row["format"], "product": row["product"], "angle": row["angle"],
            "brief": row["brief"],
            "base": base_out, "sft": sft_out, "dpo": dpo_out,
            "base_issues": check_candidate(base_out, row["voice"], row["format"]),
            "sft_issues": check_candidate(sft_out, row["voice"], row["format"]),
            "dpo_issues": check_candidate(dpo_out, row["voice"], row["format"]),
        }
        f.write(json.dumps(out_row) + "\n")
        f.flush()

        if (i + 1) % 5 == 0:
            print(f"  {i + 1}/{len(test_set)} done")

print(f"Done. Output written to {OUT_PATH}")


## 8. Rule-based compliance summary (quick objective read before the LLM judge)

In [ ]:
rows = [json.loads(l) for l in open(OUT_PATH)]

def clean_rate(rows, key):
    return sum(1 for r in rows if not r[f"{key}_issues"]) / len(rows)

print(f"{len(rows)} test prompts evaluated\n")
print(f"{'Variant':<10} {'Rule-clean rate':<18}")
for key, label in [("base", "Base"), ("sft", "SFT-only"), ("dpo", "SFT+DPO")]:
    print(f"{label:<10} {clean_rate(rows, key):.0%}")


## Next step

Download `eval_generations.jsonl` from Drive (or `!cp` it into a location
you can pull from the Colab file browser) into your local repo's `data/`
folder, then run:

```
python scripts/evaluate_and_report.py
```

That local script adds the LLM-as-judge pairwise win-rate (SFT+DPO vs.
SFT-only) and per-dimension rubric scores on top of the rule-based
numbers above, and writes the final `docs/eval_results.md` report.
